In [1]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st
import statsmodels.formula.api as smf
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import altair as alt

In [2]:
kc = pd.read_csv('kc_house_data.csv')

In [3]:
kc['date'] = pd.to_datetime(kc.date)

In [4]:
sqft_to_sqmeters = 0.09290303997

In [5]:
kc = kc.astype({'sqft_living': float, 'sqft_lot': float, 
                'sqft_above': float, 'sqft_basement': float,
                'sqft_living15': float, 'sqft_lot15': float})

In [6]:
kc.loc[:,['sqft_living', 'sqft_lot', 'sqft_above', 'sqft_basement', 'sqft_living15', 'sqft_lot15']] =\
kc.loc[:,['sqft_living', 'sqft_lot', 'sqft_above', 'sqft_basement', 'sqft_living15', 'sqft_lot15']] * sqft_to_sqmeters

In [7]:
kc.columns = ['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqm_living',
       'sqm_lot', 'floors', 'waterfront', 'view', 'condition', 'grade',
       'sqm_above', 'sqm_basement', 'yr_built', 'yr_renovated', 'zipcode',
       'lat', 'long', 'sqm_living15', 'sqm_lot15']

In [8]:
kc = kc.loc[:, ['id', 'date', 'price', 
            'floors', 'bedrooms', 'bathrooms', 
            'yr_built', 'yr_renovated',
            'waterfront', 'view', 'condition', 'grade', 
            'sqm_living', 'sqm_lot', 
            'sqm_above', 'sqm_basement',
            'sqm_living15', 'sqm_lot15',
            'lat', 'long']]

In [9]:
kc.loc[kc.yr_renovated == 0, 'yr_renovated'] = np.nan

In [10]:
kc.loc[kc.sqm_basement == 0, 'sqm_basement'] = np.nan

##### Altair renderers

In [11]:
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [12]:
alt.renderers.active

'default'

In [13]:
alt.renderers.enable('default')
# https://altair-viz.github.io/user_guide/display_frontends.html

RendererRegistry.enable('default')

##### Plotly configuration

In [14]:
base = pio.templates["simple_white"]
custom_template = base.layout.template

custom_template.layout.update(
    margin=dict(l=40, r=0, t=20, b=40),
    xaxis=dict(
        title_standoff=0,
        ticks="inside",
        showgrid=True
    ),
    yaxis=dict(
        title_standoff=0,
        ticks="inside",
        showgrid=True
    )
)

pio.templates["tight"] = custom_template
pio.templates.default = "tight"

In [15]:
pio.renderers

Renderers configuration
-----------------------
    Default renderer: 'vscode'
    Available renderers:
        ['plotly_mimetype', 'jupyterlab', 'nteract', 'vscode',
         'notebook', 'notebook_connected', 'kaggle', 'azure', 'colab',
         'cocalc', 'databricks', 'json', 'png', 'jpeg', 'jpg', 'svg',
         'pdf', 'browser', 'firefox', 'chrome', 'chromium', 'iframe',
         'iframe_connected', 'sphinx_gallery', 'sphinx_gallery_png']

In [ ]:
# px.scatter_map(kc, lat='lat', lon='long', size='sqm_living', color='price', 
#                template='plotly_dark', map_style='carto-darkmatter', 
#                color_continuous_scale='plasma', size_max=6, zoom=9.1,
#                ).update_layout(margin=dict(l=0,r=0,b=0,t=0)
#                ).show(renderer='png', width=1500, height=500)

# Histograms

In [ ]:
# px.histogram(kc, ['sqm_above', 'sqm_basement','sqm_living'],
#             color_discrete_sequence=px.colors.qualitative.T10,
#             template='simple_white', opacity=0.3,
#             range_y=[0,400], 
# ).update_traces(xbins_start=0, xbins_end=400, xbins_size=2,
#                 marker_line_width=0, marker_cornerradius=10
# ).update_layout(barmode='overlay',
#                 margin=dict(l=0,r=0,b=0,t=0), height = 330
# ).update_xaxes(title_standoff=0, ticks='inside',  showgrid=True, title_text='square meters'
# ).update_yaxes(title_standoff=0, ticks='inside',  showgrid=True)

In [ ]:
# px.histogram(kc, x='price', height=200, log_x=True
# ).update_traces(
#     xbins_size=10**4,
#     marker_cornerradius=10
# )
# # .update_xaxes(dtick='L50000') # log tick scale
# # .update_xaxes(dtick=np.log10(2)) # for base 2

In [18]:
# alt.Chart(kc, width=1400).transform_fold(
#     ['sqm_living','sqm_above', 'sqm_basement'],
#     as_=['type', 'area']
# ).mark_area(
#     clip=True,
#     opacity=0.3,
#     # line=True,
#     interpolate='step-after',
# ).encode(
#     alt.X('area:Q').bin(extent=[0,400], step=2),
#     alt.Y('count()').stack(None).scale(domainMax=500),
#     alt.Color('type:N'),
# )
#     # for mark_bar
#     # cornerRadiusEnd=10,
#     # binSpacing=0,

In [19]:
# sns.set(rc={"figure.figsize":(16,4)})
# sns.set_style('white')
# sns.histplot(kc[['sqm_above', 'sqm_basement', 'sqm_living']],
#              binrange=(0,400), binwidth=2, palette=['#1f77b4', '#ff7f0e', '#d62728'],
#              kde=False, kde_kws={'clip':(0,400)},
#              multiple='layer', element='step', alpha=0.3, linewidth=0)

# Scatterplots

In [21]:
# px.scatter(kc.loc[kc.price < 10**6], x='date', y='price', color='bedrooms',
#            template='simple_white', opacity=0.4, width=1600, height=500,
#            range_x=(['2014-05','2015-06']), range_y=[0,1*10**6],
#            range_color=([0,5]),
#            color_continuous_scale=px.colors.sequential.Inferno,
# ).update_traces(marker_size=14
# ).update_layout(margin=dict(l=50, r=0, t=20, b=0)
# ).update_xaxes(title_standoff=0, ticks='inside', showgrid=True,
# ).update_yaxes(title_standoff=0, ticks='inside', showgrid=True)

In [20]:
# Plotly themes: 
#     'ggplot2', 'seaborn', 'plotly', None    # - ggplot2 = seaborn, plotly = None
#     'simple_white',                         # - just white, no grid, black axes
#     'plotly_white',                         # - white, thin grids and axes
#     'presentation',                         # - the same but bigger fonts, bordered legend
#     'xgridoff','ygridoff', 'gridon',        # - = plotly white
#     'plotly_dark']                          # - inverted plotly white

In [1]:
# sns.set(rc={"figure.figsize":(14,7)})
# sns.set_style('white')
# sns.scatterplot(kc.loc[kc.price<10**6], x='date', y='price', hue='bedrooms',
#                 palette='inferno', s=100, alpha=0.4, hue_norm=(0,5), linewidth = 0)
# plt.legend(loc=[1,0], markerscale=1)

In [22]:
# alt.Chart(kc.loc[kc.price<10**6], width=1400, height=500).mark_circle(tooltip=True,
#     size=200, opacity=0.4, clip=True).encode(
#     alt.X('date', type='temporal').axis(grid=False),
#     alt.Y('price', type='quantitative').axis(grid=False),
#     alt.Color(field ='bedrooms', type='quantitative').scale(
#         domain=[0,5], scheme='inferno')).properties(background='white')

In [23]:
# alt.renderers.active

In [24]:
# .display(renderer='svg')
# jupyter - 6.7s (less blurry)
# default - 0.5s (blurry if os scaling!= monitor scaling)
# svg - 1s (no blur)
# png - 2s (less blurry, no white background needed)
# mimetype - 0.5s (less blurry, needs .properties(background='white')
# seaborn is also blurry when scalings differ

# Scatter Matrices

In [27]:
# alt.Chart(kc).mark_circle(
#     size=20, opacity=0.3, tooltip=True
# ).encode(
#     alt.X(alt.repeat("column"), type='quantitative').scale(zero=False),
#     alt.Y(alt.repeat("row"), type='quantitative').scale(zero=False),
#     alt.Color('condition:N').scale(scheme='tableau10')
# ).properties(
#     width=225,
#     height=225
# ).repeat(
#     row=['price', 'grade', 'sqm_living', 'sqm_lot', 'yr_built'],
#     column=['price', 'grade', 'sqm_living', 'sqm_lot', 'yr_built']
# ).configure_facet(spacing=0
# ).configure_legend(symbolOpacity=1)

In [26]:
# jupyter - 9.4s
# jupyter (, offline=True) - 9.3s
# svg - 9.6s
# png - 23s
# default(html) - 0.5s (blurry when os scaling different from monitor)
# mimetype - 0.5s (png, transparent background, needs .properties(background='white')

In [28]:
# px.scatter_matrix(kc, dimensions=['price', 'grade', 'sqm_living', 'sqm_lot', 'yr_built'], color='condition',
#                   template='plotly_white', opacity=0.3,
#                   color_continuous_scale=px.colors.qualitative.T10[:5],
#                   height=1530, width=1530
# ).update_traces(marker_size=5.1, showlowerhalf=True
# ).update_layout(margin=dict(l=0, r=0, t=20, b=0)
# ).update_xaxes(title_standoff=0, ticks='inside'
# ).update_yaxes(title_standoff=0, ticks='inside')

In [1]:
# sns.set(rc={"figure.figsize":(18,18)})
# sns.set_theme(style="ticks")
# sns.pairplot(kc, hue="condition", vars=['price', 'grade', 'sqm_living', 'sqm_lot', 'yr_built'],
#             #  kind='reg', plot_kws={'ci' : None, 'scatter_kws' : {'s': 1, 'alpha' : 0.2}, 'line_kws' : {'lw': 1}},
#              kind='scatter', plot_kws={'s': 10, 'alpha': 0.2, 'linewidth' : 0},
#              palette='tab10')

In [ ]:
# facets = []
# features = ['price', 'grade', 'sqm_living', 'yr_built']
# color = 'condition'

# for feature1 in features:
#     for feature2 in features:
#         if feature1 == feature2:
#             chart = alt.Chart(kc).mark_area(tooltip=True, line=True).encode(
#                 x=alt.X(f"{feature1}:Q").bin(step=1).axis(format='~s').stack('zero'), 
#                 y=alt.Y("count()").axis(format='~s'),
#                 # color=alt.Color(f"{color}:N")
#             )
#         else:
#             chart = alt.Chart(kc).mark_circle(tooltip=True).encode(
#                 x=alt.X(f"{feature2}:Q").axis(format='~s'),  
#                 y=alt.Y(f"{feature1}:Q").axis(format='~s'),  
#                 color=alt.Color(f"{color}:N")
#             )
#         facets.append(chart)

# figure = alt.vconcat(*[alt.hconcat(*facets[i:i+len(features)]) for i in range(0, len(facets), len(features))])
# figure = figure.properties(title="Pairplot")
# figure

# Datetime

In [ ]:
# Change column type to datetime64[ns] for column: 'date'
# kc = kc.astype({'date': 'datetime64[s]'})
# kc.date.str[:8].astype('datetime64[s]')[0] # same result
# kc['date'] = pd.to_datetime(kc.date)

In [30]:
# dates_limits = mpl.dates.date2num(np.datetime64(kc.date.min())) - 1,\
#                mpl.dates.date2num(np.datetime64(kc.date.max()))
# dates_range = mpl.dates.date2num(
#     pd.date_range(start=kc.date.min(), end=kc.date.max(), freq='W-MON'))

# fig, ax = plt.subplots(figsize=(18,4))
# sns.set_style('whitegrid')
# sns.histplot(kc.date, element='step', binwidth=1, alpha=0.5,
#              binrange=(dates_limits)).set_xlim(dates_limits)

# ax.xaxis.set_major_formatter(mpl.dates.DateFormatter('%d-%b-%Y'))
# ax.xaxis.set_ticks_position('top')
# plt.xticks(dates_range, rotation=90)
# plt.show()

In [ ]:
# alt.Chart(kc, width=1500).mark_bar(
#     tooltip=True,
#     opacity=1,
#     cornerRadiusEnd=10,
# ).encode(
#     alt.X('date:T').axis(
#         labelAngle=-73,
#         format='%e %b %Y'),
#     alt.Y('count()'),
# )

In [ ]:
# px.histogram(kc, 'date', height=400
# ).update_traces(xbins_start=kc.date.min(),
#                 xbins_end=kc.date.max(),
#                 xbins_size='D',
#                 marker_cornerradius=20
# ).update_xaxes(dtick=7*24*60*60*1000,
#                minor_dtick=24*60*60*1000,
#                minor_showgrid=True,
#                tickangle=-65,
#                gridcolor='#bbb',
#                ticks='outside',
#                tick0='2014-05-05'
# ).update_yaxes(gridcolor='#bbb'
# ).update_layout(margin=dict(b=70))

# Regression

In [43]:
m = smf.ols('price ~ sqm_living', data = kc)
m.fit(cov_type='HC1').summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  price   R-squared:                       0.493
Model:                            OLS   Adj. R-squared:                  0.493
Method:                 Least Squares   F-statistic:                     2469.
Date:                Sat, 20 Sep 2025   Prob (F-statistic):               0.00
Time:                        22:24:22   Log-Likelihood:            -3.0027e+05
No. Observations:               21613   AIC:                         6.005e+05
Df Residuals:                   21611   BIC:                         6.006e+05
Df Model:                           1                                         
Covariance Type:                  HC1                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -4.358e+04   1.08e+04     -4.036      0.000   -6.47e+04   -2.24e+04
sqm_living  3020.6069     60.786     49.693      0.000    2901.469    3139.745
==============================================================================
Omnibus:                    14832.490   Durbin-Watson:                   1.983
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           546444.713
Skew:                           2.824   Prob(JB):                         0.00
Kurtosis:                      26.977   Cond. No.                         523.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
"""

##### In-function regression

In [2]:
# (
# alt.Chart(kc).mark_circle(
#     opacity=0.2
# ).encode(
#     alt.X('sqm_living:Q'),
#     alt.Y('grade:Q')
# ) +
# alt.Chart(kc).transform_regression(
#     on='sqm_living',
#     regression='grade',
#     # as=    # names for x, y
#     # extent=[,]    #on x axis
#     # groupby=['condition']
#     method='quad',
#     # order=3
#     # params=True
# ).mark_line().encode(
#     alt.X('sqm_living:Q'),
#     alt.Y('grade:Q'),
#     # alt.Color('condition:N')
#     # text='rSquared:N'
# )
# )

In [34]:
# # poly   : y = a + b * x + … + k * x^(order)
# # quad   : y = a + b * x + c * x^2           -> poly of order 2
# # linear : y = a + b * x                     -> poly of order 1
# # 
# # power  : y = a * x^b
# # exp    : y = a * e^(b * x)
# # log    : y = a + b * log(x)

In [ ]:
# sns.lmplot(kc, y='price', x='bathrooms', order=2,
#            aspect=2, height=6, markers='D',
#            x_estimator=np.mean,
#            x_ci='ci', # 'sd', 'ci', specific ci, None - only for means
#            # alongside x_bins, these do not affect the regression or its errorband, they sepparately aggregate the scatterplot
#            ci=95, # affects both the ci of regression and aggergation
#            scatter_kws={'s': 3, 'alpha':0.3}, line_kws={'linewidth':1})

In [53]:
# sns.residplot(kc, x='sqm_living', y='price', lowess=True, robust=True, 
#               scatter_kws={'s': 10, 'alpha':0.1}, line_kws={'linewidth':2})

##### Aggregation

In [35]:
# (
# alt.Chart(kc).mark_circle(opacity=0.2).encode(
#     alt.X('grade:O'),
#     alt.Y('price:Q')
# ) +

# alt.Chart(kc).mark_line(width=2).encode(
#     alt.X('grade:O'),
#     alt.Y('mean(price):Q')
# ) +

# alt.Chart(kc).mark_errorband(    # only works for means, not regression
#     borders=True,
#     extent='stdev'
# ).encode(
#     alt.X('grade:O'),
#     alt.Y('price:Q')
# )
# ).properties(width=1500)

In [ ]:
# sns.set(rc={"figure.figsize":(20,4)})
# sns.set_style('whitegrid')
# sns.lineplot(kc, y='price', x='yr_built',
#              errorbar= 'ci') # will not look the same, unless a unique seed is mentioned
# sns.lineplot(kc, y='price', x='yr_built', linewidth=0, color='#ff7f0e',
#              errorbar= ('se', 2)) # 2*SE = aprox the same as 'ci'

##### External error data

In [36]:
m.fit().get_prediction().summary_frame()

,mean,mean_se,mean_ci_lower,mean_ci_upper,obs_ci_lower,obs_ci_upper
0,6.778444,0.007240,6.764254,6.792635,5.288246,8.268643
1,8.135280,0.005861,8.123791,8.146769,6.645105,9.625455
2,6.378227,0.009008,6.360571,6.395882,4.887991,7.868462
3,7.539834,0.005215,7.529612,7.550056,6.049668,9.030000
4,7.266515,0.005640,7.255460,7.277570,5.776343,8.756687
...,...,...,...,...,...,...
21608,7.120094,0.006027,7.108280,7.131908,5.629916,8.610271
21609,7.881483,0.005331,7.871034,7.891933,6.391316,9.371651
21610,6.622262,0.007897,6.606784,6.637740,5.132051,8.112473
21611,7.188423,0.005835,7.176987,7.199860,5.698249,8.678598


In [ ]:
# (
# alt.Chart(    # scatterplot
#     kc.merge(m.fit().get_prediction().summary_frame(), how = 'left', left_index=True, right_index=True)
# ).mark_circle(
#     opacity=0.2, tooltip=True, size=80
# ).encode(
#     alt.X('sqm_living:Q'),
#     alt.Y('price:Q')
# ) +

# alt.Chart(    # line, predicted values
#     kc.merge(m.fit().get_prediction().summary_frame(), how = 'left', left_index=True, right_index=True)
# ).mark_line(
#         width=1, color='black'
# ).encode(
#     alt.X('sqm_living:Q'),
#     alt.Y('mean:Q')
# ) +

# alt.Chart(    # band, prediction interval - like mark_area, but which also has interpolation
#     kc.merge(m.fit().get_prediction().summary_frame(), how = 'left', left_index=True, right_index=True)
# ).mark_errorband(
#     borders=True, color='magenta', opacity=0.2
# ).encode(
#     alt.X('sqm_living:Q'),
#     alt.Y('obs_ci_lower:Q'),
#     alt.Y2('obs_ci_upper:Q')
# ) +

# alt.Chart(    # band, confidence interval, calculated from mean (predicted values) + SE
#     kc.merge(m.fit().get_prediction().summary_frame(), how = 'left', left_index=True, right_index=True)
# ).mark_errorband(
#     borders=True,
# ).encode(
#     alt.X('sqm_living:Q'),
#     alt.Y('mean:Q'),
#     alt.YError('mean_se:Q')
# )
# ).properties(width=1500, height=1500)

In [ ]:
# unstable

# fig = px.scatter(kc.merge(m.fit().get_prediction().summary_frame(), how = 'left', left_index=True, right_index=True), 
#            x='sqm_living', y='price', #trendline="ols", 
#            opacity=0.3, color='grade', height=1000,
#            color_continuous_scale=px.colors.sequential.YlOrRd)

# fig.add_trace(
# px.area(kc.merge(m.fit().get_prediction().summary_frame(), how = 'left', left_index=True, right_index=True),
#         x='sqm_living',
#         y='obs_ci_lower',
#         # y=['obs_ci_lower', 'obs_ci_upper'],
#         ).update_traces(
#             fillcolor='cyan', stackgroup="", alignmentgroup=1
#             ).data[0])
# fig.add_trace(
# px.area(kc.merge(m.fit().get_prediction().summary_frame(), how = 'left', left_index=True, right_index=True),
#         x='sqm_living',
#         y='mean_ci_lower',
#         # y=['mean_ci_lower', 'mean_ci_upper'],
#         ).update_traces(
#             fill='tonexty', fillcolor='green', stackgroup="", alignmentgroup=1
#             ).data[0])
# fig.add_trace(
# px.area(kc.merge(m.fit().get_prediction().summary_frame(), how = 'left', left_index=True, right_index=True),
#         x='sqm_living',
#         y='mean_ci_upper',
#         # y=['mean_ci_lower', 'mean_ci_upper'],
#         ).update_traces(
#             fill='tonexty', fillcolor='green', stackgroup="", alignmentgroup=1
#             ).data[0])
# fig.add_trace(
# px.area(kc.merge(m.fit().get_prediction().summary_frame(), how = 'left', left_index=True, right_index=True),
#         x='sqm_living',
#         y='obs_ci_upper',
#         # y=['obs_ci_lower', 'obs_ci_upper'],
#         ).update_traces(
#             fillcolor='cyan', stackgroup="", alignmentgroup=1,
#             ).data[0])

# fig.show()

Residplots

In [ ]:
# (

# (
# alt.Chart(    # like residplot
#     kc.merge(m.fit().get_prediction().summary_frame(), how = 'left', left_index=True, right_index=True),
#     width=700, height=700
# ).transform_calculate(
#     residuals=alt.datum.price - alt.datum.mean
# ).mark_circle(
#     opacity=0.2, tooltip=True
# ).encode(
#     alt.X('sqm_living:Q').axis(format='~s'),
#     alt.Y('residuals:Q').axis(format='~s'),
#     alt.Color('grade:N')
# ) + 
# alt.Chart(kc.merge(m.fit().get_prediction().summary_frame(), how = 'left', left_index=True, right_index=True),
#     width=700, height=700
# ).mark_rule(strokeDash=[2, 2]).encode(y=alt.datum(0)
# )

# |

# alt.Chart(    # residuals against fitted values, useful for multiple predictors
#     kc.merge(m.fit().get_prediction().summary_frame(), how = 'left', left_index=True, rightpx.histogram(kc, 'date', height=400
#     width=700, height=700
# ).transform_calculate(
#     residuals=alt.datum.price - alt.datum.mean
# ).mark_circle(
#     opacity=0.2, tooltip=True
# ).encode(
#     alt.X('mean:Q').axis(format='~s'),
#     alt.Y('residuals:Q').axis(format='~s'),
#     alt.Color('grade:N')
# ) + 
# alt.Chart(kc.merge(m.fit().get_prediction().summary_frame(), how = 'left', left_index=True, right_index=True),
#     width=700, height=700
# ).mark_rule(strokeDash=[2, 2]).encode(y=alt.datum(0)
# )

# )
# &
# (
# alt.Chart(    # residual against predicted values
#     kc.merge(m.fit().get_prediction().summary_frame(), how = 'left', left_index=True, right_index=True),
#     width=700, height=700
# ).transform_calculate(
#     residuals=alt.datum.price - alt.datum.mean
# ).mark_circle(
#     opacity=0.2, tooltip=True
# ).encode(
#     alt.X('price:Q').axis(format='~s'),
#     alt.Y('residuals:Q').axis(format='~s'),
#     alt.Color('grade:N')
#     ) + 
# alt.Chart(kc.merge(m.fit().get_prediction().summary_frame(), how = 'left', left_index=True, right_index=True),
#     width=700, height=700
# ).mark_rule(strokeDash=[2, 2]).encode(y=alt.datum(0)
# ) 

# |

# alt.Chart(    # original regression
#     kc.merge(m.fit().get_prediction().summary_frame(), how = 'left', left_index=True, right_index=True),
#     width=700, height=700
# ).transform_calculate(
#     residuals=alt.datum.price - alt.datum.mean
# ).mark_circle(
#     opacity=0.2, tooltip=True
# ).encode(
#     alt.X('sqm_living:Q').axis(format='~s'),
#     alt.Y('price:Q').axis(format='~s'),
#     alt.Color('grade:N')
# ) +
# alt.Chart(    # line, predicted values
#     kc.merge(m.fit().get_prediction().summary_frame(), how = 'left', left_index=True, right_index=True),
#     width=700, height=700
# ).mark_line(
#         width=1, color='black', strokeDash=[2, 2]
# ).encode(
#     alt.X('sqm_living:Q'),
#     alt.Y('mean:Q')
# )
# )

# ).configure_legend(symbolOpacity=1)

QQplots

In [ ]:
m.fit().resid.std()

In [2]:
# sns.set(rc={"figure.figsize":(5,5)})
# sm.qqplot(data = m.fit().resid[m.fit().resid < 16 * m.fit().resid.std()],
#           line ='45', fit=True)
# plt.show()

In [4]:
# sns.set(rc={"figure.figsize":(20,2)})
# sns.histplot(st.zscore(m.fit().resid), element='step', alpha=0.5)
# sns.histplot(st.zscore(np.random.normal(0, m.fit().resid.std(), 21613)),
#              element='step', alpha=0.2, color='#ff7f0e').set(xticks=range(-5, 16, 1))